In [ ]:
import os
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

import matplotlib.pyplot as plt

torch.manual_seed(8008135)

NOTEBOOK_DIR = Path.cwd()
CODE_DIR = NOTEBOOK_DIR.parent

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("CODE_DIR:", CODE_DIR)
print("CODE_DIR contents:", os.listdir(CODE_DIR))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Device set to {device}")

if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

In [ ]:
from Datasets.Sudoku_DataLoader import get_loaders

from HRM_Model.HRM_Model import HRM_ACT
from HRM_Model.HRM_Components import Encoder, HighLevel, LowLevel, Head, QHead
from HRM_Model.HRM_ACT_Train import train_hrm_act, evaluate_hrm_act

from Utils.schedules import cosine_schedule_with_warmup_lr_lambda
from Utils.checkpointing import load_checkpoint

In [ ]:
train_size = 2**18
test_size = 2**15
batch_size = 2**7

train_dataloader, val_dataloader = get_loaders(
    train_size=train_size,
    test_size=test_size,
    batch_size=batch_size,
)

In [ ]:
d_model = 512
M_max = 8
N = 2
T = 2
n_layers = 4
n_heads = 8
vocab_size = 10
dropout = 0.2
epsilon = 0.1

lr = 5e-5
min_lr_ratio = 0.1
lr_warmup = 0.05
beta1 = 0.9
beta2 = 0.95
weight_decay = 0.1
num_epochs = 5

checkpoint_dir = "checkpoints_act"
pretrained_path = "checkpoints/hrm_best_34M.pt"

In [ ]:
high_level = HighLevel(
    d_model=d_model,
    n_layers=n_layers,
    n_heads=n_heads,
    intermediate_size=4 * d_model,
    dropout=dropout,
)

low_level = LowLevel(
    d_model=d_model,
    n_layers=n_layers,
    n_heads=n_heads,
    intermediate_size=4 * d_model,
    dropout=dropout,
)

encoder = Encoder(
    vocab_size=vocab_size,
    d_model=d_model,
)

head = Head(
    d_model=d_model,
    vocab_size=vocab_size,
)

q_head = QHead(
    d_model=d_model,
)

HRM_ACT_model = HRM_ACT(
    L_module=low_level,
    H_module=high_level,
    encoder=encoder,
    head=head,
    q_head=q_head,
    M=M_max,
    N=N,
    T=T,
    max_len=81,
    d_model=d_model,
    epsilon=epsilon,
).to(device)

if pretrained_path is not None and os.path.exists(pretrained_path):
    ckpt = torch.load(pretrained_path, map_location=device)
    state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    missing, unexpected = HRM_ACT_model.load_state_dict(state, strict=False)
    print(f"Loaded pretrained backbone from {pretrained_path}")
    print(f"Missing keys: {missing}")
    print(f"Unexpected keys: {unexpected}")
else:
    print("Training ACT from scratch")

print(
    "Number of trainable parameters:",
    f"{sum(p.numel() for p in HRM_ACT_model.parameters() if p.requires_grad):,}",
)

In [ ]:
optimizer = optim.AdamW(
    HRM_ACT_model.parameters(),
    lr=lr,
    betas=(beta1, beta2),
    weight_decay=weight_decay,
)

num_training_steps = len(train_dataloader) * num_epochs * M_max
num_warmup_steps = int(lr_warmup * num_training_steps)

scheduler = LambdaLR(
    optimizer,
    lr_lambda=lambda step: cosine_schedule_with_warmup_lr_lambda(
        step,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
        min_ratio=min_lr_ratio,
    ),
)

print("num_training_steps:", num_training_steps)
print("num_warmup_steps:", num_warmup_steps)

In [ ]:
HRM_ACT_model, best_metric, history = train_hrm_act(
    model=HRM_ACT_model,
    train_loader=train_dataloader,
    optimizer=optimizer,
    loss_fn=nn.CrossEntropyLoss(ignore_index=-100),
    device=device,
    scheduler=scheduler,
    num_epochs=num_epochs,
    M_max=M_max,
    checkpoint_dir=checkpoint_dir,
    checkpoint_every=1,
    validate_every=1,
    val_loader=val_dataloader,
)

HRM_ACT_model.eval()

print("Best ACT val board accuracy:", best_metric)

In [ ]:
train_steps = history["step"]
train_loss = history["train_loss"]
ce = history["ce"]
bce = history["bce"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_steps, ce, label="CE", linewidth=1)
axes[0].plot(train_steps, bce, label="BCE (Q-head)", linewidth=1)
axes[0].set_xlabel("Training step")
axes[0].set_ylabel("Loss")
axes[0].set_title("ACT Loss Components")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(train_steps, history["mean_m_at_halt"], label="mean M at halt", linewidth=1)
axes[1].plot(train_steps, history["halt_acc"], label="halt accuracy", linewidth=1)
axes[1].axhline(M_max, color="red", linestyle="--", alpha=0.5, label=f"M_max={M_max}")
axes[1].set_xlabel("Training step")
axes[1].set_title("ACT Halt Behaviour")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
HRM_ACT_model.eval()

print(f"{'M_max':>6} | {'token_acc':>10} | {'board_acc':>10} | {'mean_m':>7}")
print("-" * 45)

for mmax in [2, 4, 8, 16]:
    tok, board, mean_m = evaluate_hrm_act(
        HRM_ACT_model,
        val_dataloader,
        device,
        M_max=mmax,
    )
    print(f"{mmax:>6} | {tok * 100:>9.2f}% | {board * 100:>9.2f}% | {mean_m:>7.2f}")